# LangChain Setup (OpenRouter)

This notebook verifies your OpenRouter setup and establishes the core execution patterns you will reuse in later notebooks.

**Goals**
- Load `.env` and resolve your model from `OPENROUTER_MODEL`.
- Instantiate `ChatOpenRouter`.
- Use `invoke`, `batch`, and `stream`.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4o-mini")
if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY is missing. Add it to .env")

print(f"Using model: {OPENROUTER_MODEL}")

Using model: google/gemini-3.5-flash


In [3]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model=OPENROUTER_MODEL,
    api_key=OPENROUTER_API_KEY,
    temperature=0.2,
    max_retries=2,
    max_tokens=2048,
)

model

ChatOpenRouter(output_version=None, client=<openrouter.sdk.OpenRouter object at 0x0000021B9BD0FCB0>, openrouter_api_key=SecretStr('**********'), openrouter_api_base=None, app_url='https://docs.langchain.com', app_title='LangChain', model_name='google/gemini-3.5-flash', temperature=0.2, max_tokens=2048, model_kwargs={}, session_id=None)

## Basic invoke
`invoke` returns an AIMessage for chat models.

In [4]:
response = model.invoke("Explain LCEL in one sentence.")
response

AIMessage(content='**LangChain Expression Language (LCEL)** is a declarative, pipe-based syntax that simplifies composing, customizing, and productionizing complex LLM chains with built-in support for streaming, parallel execution, and async operations.', additional_kwargs={'reasoning_content': "**Composing LCEL Expression**\n\nI'm currently focusing on crafting a concise, single-sentence explanation of LangChain Expression Language (LCEL), aiming to capture its core functionality for an expert audience.\n\n**Refining LCEL Definition**\n\nI'm honing in on a precise, single-sentence description for LCEL, specifically targeting an expert audience. My latest drafts emphasize its declarative, pipe-based nature and its benefits like streaming and async support.\n\n", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'google-gemini-v1', 'index': 0, 'text': "**Composing LCEL Expression**\n\nI'm currently focusing on crafting a concise, single-sentence explanation of LangChain Express

## Batch invoke
Batching works on the same interface.

In [5]:
batch_responses = model.batch(["Define RAG", "Define a retriever"])
[r.content for r in batch_responses]

['**RAG** stands for **Retrieval-Augmented Generation**. \n\nIt is an artificial intelligence (AI) framework that improves the quality and accuracy of Large Language Model (LLM) responses by grounding the model on external, authoritative sources of information before generating a response.\n\nIn simple terms, RAG gives an AI model an **"open-book exam"** instead of forcing it to rely solely on its memory.\n\n---\n\n### The Analogy: Closed-Book vs. Open-Book\n* **Without RAG (Closed-Book):** You ask an LLM a question. It relies only on the data it was trained on (which might be outdated or missing your private company data). If it doesn\'t know the answer, it might make something up (hallucinate).\n* **With RAG (Open-Book):** You ask the LLM a question. The system first searches a library of documents (your database, PDFs, or the web) for the correct information, hands those documents to the LLM, and asks the LLM to write an answer based *only* on those documents.\n\n---\n\n### How RAG 

## Streaming
Streaming yields partial chunks as they are generated.

In [6]:
for chunk in model.stream("Give 3 bullet points about LCEL."):
    print(chunk.content, end="")

Here are 3 key points about LCEL (LangChain Expression Language):

*   **Declarative Chaining Syntax:** LCEL uses a simple, declarative syntax (utilizing the pipe operator `|`) to easily compose and chain LangChain components together, making complex LLM workflows highly readable and modular.
*   **Production-Ready Features Out-of-the-Box:** It automatically provides advanced capabilities like streaming, asynchronous execution, batch processing, and parallel step execution without requiring any custom code.
*   **Built-in Reliability and Tracing:** LCEL includes native support for fallbacks (to handle API failures) and integrates seamlessly with LangSmith for automatic, step-by-step tracing and debugging of your chains.

## Model capability note
Tool calling and structured output depend on the chosen OpenRouter model. If a model does not support tool calling, the tool-related sections in later notebooks will fail gracefully.